In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


# USB SSD vs Internal SSD Benchmark
Compare Arina 4D-STEM loading speed between internal SSD and USB SSD (Crucial X10).
The bottleneck is Metal GPU decompression, not disk IO — so USB should be nearly as fast.

**Dataset**: SnMoS2s — 512×512 scan, 192×192 detector, 2.8 GB compressed on disk.

In [2]:
import time
import gc
import torch
import numpy as np
from quantem.widget import IO, Show2D, Show4DSTEM

In [3]:
INTERNAL = "/Users/macbook/data/guoliang/Show4DSTEM/20260208/SnMoS2s_001_master.h5"
USB = "/Volumes/Crucial X10/arina_bench/SnMoS2s_001_master.h5"
DET_BINS = [4, 2]

## Warmup
First load primes the Metal GPU pipeline (shader compilation, buffer allocation).

In [4]:
r = IO.arina_file(INTERNAL, det_bin=8)
print(f"warmup shape: {r.data.shape}")
del r; gc.collect(); torch.mps.empty_cache()

load_arina: 262144 frames, det (192,192) → (24,24), 1.51s
hot pixel filter: 4 pixels zeroed (5.0σ threshold)
warmup shape: (512, 512, 24, 24)


## Benchmark: Internal SSD vs USB SSD
Each test loads the same 2.8 GB compressed dataset, decompresses on GPU, and bins the detector.

In [5]:
results = []
for det_bin in DET_BINS:
    for label, path in [("Internal SSD", INTERNAL), ("USB SSD", USB)]:
        gc.collect(); torch.mps.empty_cache()
        t0 = time.perf_counter()
        r = IO.arina_file(path, det_bin=det_bin)
        elapsed = time.perf_counter() - t0
        mem_gb = r.data.nbytes / 1e9
        results.append((label, det_bin, r.data.shape, elapsed, mem_gb))
        print(f"{label:>15s}  det_bin={det_bin}  {r.data.shape}  {elapsed:.2f}s  ({mem_gb:.1f} GB)")
        del r; gc.collect(); torch.mps.empty_cache()

load_arina: 262144 frames, det (192,192) → (48,48), 1.46s
hot pixel filter: 4 pixels zeroed (5.0σ threshold)
   Internal SSD  det_bin=4  (512, 512, 48, 48)  1.73s  (2.4 GB)
load_arina: 262144 frames, det (192,192) → (48,48), 1.43s
hot pixel filter: 4 pixels zeroed (5.0σ threshold)
        USB SSD  det_bin=4  (512, 512, 48, 48)  1.44s  (2.4 GB)
load_arina: 262144 frames, det (192,192) → (96,96), 2.39s
hot pixel filter: 4 pixels zeroed (5.0σ threshold)
   Internal SSD  det_bin=2  (512, 512, 96, 96)  2.76s  (9.7 GB)
load_arina: 262144 frames, det (192,192) → (96,96), 2.77s
hot pixel filter: 4 pixels zeroed (5.0σ threshold)
        USB SSD  det_bin=2  (512, 512, 96, 96)  2.78s  (9.7 GB)


## Results

In [6]:
print(f"{'Storage':>15s}  {'det_bin':>7s}  {'Shape':>24s}  {'Time':>6s}  {'Output':>8s}")
print("-" * 70)
for label, det_bin, shape, elapsed, mem_gb in results:
    shape_str = "x".join(str(s) for s in shape)
    print(f"{label:>15s}  {det_bin:>7d}  {shape_str:>24s}  {elapsed:>5.2f}s  {mem_gb:>6.1f} GB")

        Storage  det_bin                     Shape    Time    Output
----------------------------------------------------------------------
   Internal SSD        4             512x512x48x48   1.73s     2.4 GB
        USB SSD        4             512x512x48x48   1.44s     2.4 GB
   Internal SSD        2             512x512x96x96   2.76s     9.7 GB
        USB SSD        2             512x512x96x96   2.78s     9.7 GB


## Visual verification
Load from USB and display — confirm data integrity matches internal SSD.

In [7]:
r_usb = IO.arina_file(USB, det_bin=2)
print(r_usb)

load_arina: 262144 frames, det (192,192) → (96,96), 2.29s
hot pixel filter: 4 pixels zeroed (5.0σ threshold)
IOResult
  shape:      512 x 512 x 96 x 96
  dtype:      float32
  memory:     9.0 GB
  title:      SnMoS2s_001
  metadata:   41 fields


In [8]:
Show4DSTEM(r_usb, title="SnMoS2 from USB SSD")

  to cpu: 0.01s (9.7 GB)
  auto_detect_center: 0.40s
  virtual image + frame: 0.01s
Show4DSTEM: 512x512x96x96 cpu, 0.68s total


Show4DSTEM(shape=(512, 512, 96, 96), sampling=(1.0 Å, 1.0 px), pos=(256, 256), title='SnMoS2 from USB SSD')

In [ ]:
Show2D(
    r_usb.data.sum(axis=(2, 3)),
    title="Virtual BF — USB SSD",
    log_scale=True,
    show_fft=True,
)

## Cleanup

In [ ]:
del r_usb; gc.collect(); torch.mps.empty_cache()
print(f"MPS memory: {torch.mps.current_allocated_memory() / 1e9:.1f} GB allocated")